# Comparaison des Modèles - Clean Original vs Clean_2

Ce notebook compare les performances des modèles entraînés sur les deux datasets.

## 1. Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_data, get_features_and_target
from src.preprocessing import NutriscorePreprocessor
from src.feature_engineering import create_engineered_features
from src.models import create_random_forest_model, create_xgboost_model
from src.training import train_model

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Imports réussis")

## 2. Fonction d'Entraînement

In [ ]:
def train_and_evaluate(data_path, dataset_name):
    print(f"\n{'='*60}")
    print(f"ENTRAÎNEMENT - {dataset_name}")
    print(f"{'='*60}")
    
    # Chargement
    df = load_data(str(data_path))
    print(f"\n📊 Dataset: {df.shape[0]:,} produits")
    
    # Features
    X, y = get_features_and_target(df)
    print(f"Distribution Nutri-Score:")
    print(y.value_counts().sort_index())
    
    # Feature engineering
    X_eng = create_engineered_features(X)
    print(f"\n🔧 Features: {X_eng.shape[1]}")
    
    # Preprocessing
    preprocessor = NutriscorePreprocessor()
    X_train, X_test, y_train, y_test = preprocessor.fit_transform(
        X_eng, y, test_size=0.3, random_state=42
    )
    print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
    
    # XGBoost
    print(f"\n🚀 XGBoost...")
    xgb_model = create_xgboost_model()
    xgb_model, xgb_metrics = train_model(
        xgb_model, X_train, y_train, X_test, y_test, use_sample_weights=True
    )
    
    # Random Forest
    print(f"\n🌲 Random Forest...")
    rf_model = create_random_forest_model()
    rf_model, rf_metrics = train_model(
        rf_model, X_train, y_train, X_test, y_test, use_sample_weights=False
    )
    
    # Meilleur modèle
    best_model = rf_model if rf_metrics['test_f1'] > xgb_metrics['test_f1'] else xgb_model
    best_name = 'Random Forest' if rf_metrics['test_f1'] > xgb_metrics['test_f1'] else 'XGBoost'
    best_metrics = rf_metrics if rf_metrics['test_f1'] > xgb_metrics['test_f1'] else xgb_metrics
    
    print(f"\n🏆 Meilleur: {best_name}")
    
    return {
        'dataset_name': dataset_name,
        'n_products': df.shape[0],
        'n_features': X_eng.shape[1],
        'distribution': y.value_counts().sort_index(),
        'xgb_metrics': xgb_metrics,
        'rf_metrics': rf_metrics,
        'best_name': best_name,
        'best_metrics': best_metrics
    }

## 3. Entraînement sur Clean Original

In [ ]:
data_path_1 = Path.cwd().parent / 'data' / 'openfoodfacts_clean.csv'
results_1 = train_and_evaluate(data_path_1, 'Clean Original')

## 4. Entraînement sur Clean_2

In [ ]:
data_path_2 = Path.cwd().parent / 'data' / 'openfoodfacts_clean_2.csv'
results_2 = train_and_evaluate(data_path_2, 'Clean_2')

## 5. Comparaison des Résultats

In [ ]:
print("\n" + "="*60)
print("COMPARAISON DES MODÈLES")
print("="*60)

comparison_df = pd.DataFrame({
    'Métrique': [
        'Dataset',
        'Nombre de produits',
        'Features',
        'Meilleur modèle',
        'Accuracy Train',
        'Accuracy Test',
        'F1-Score Train',
        'F1-Score Test',
        'Temps (s)'
    ],
    'Clean Original': [
        results_1['dataset_name'],
        f"{results_1['n_products']:,}",
        results_1['n_features'],
        results_1['best_name'],
        f"{results_1['best_metrics']['train_accuracy']:.4f}",
        f"{results_1['best_metrics']['test_accuracy']:.4f}",
        f"{results_1['best_metrics']['train_f1']:.4f}",
        f"{results_1['best_metrics']['test_f1']:.4f}",
        f"{results_1['best_metrics']['training_time']:.2f}"
    ],
    'Clean_2': [
        results_2['dataset_name'],
        f"{results_2['n_products']:,}",
        results_2['n_features'],
        results_2['best_name'],
        f"{results_2['best_metrics']['train_accuracy']:.4f}",
        f"{results_2['best_metrics']['test_accuracy']:.4f}",
        f"{results_2['best_metrics']['train_f1']:.4f}",
        f"{results_2['best_metrics']['test_f1']:.4f}",
        f"{results_2['best_metrics']['training_time']:.2f}"
    ]
})

print(f"\n{comparison_df.to_string(index=False)}")

## 6. Analyse des Différences

In [ ]:
print("\n" + "="*60)
print("DIFFÉRENCES")
print("="*60)

diff_products = results_2['n_products'] - results_1['n_products']
diff_acc = results_2['best_metrics']['test_accuracy'] - results_1['best_metrics']['test_accuracy']
diff_f1 = results_2['best_metrics']['test_f1'] - results_1['best_metrics']['test_f1']
diff_time = results_2['best_metrics']['training_time'] - results_1['best_metrics']['training_time']

print(f"\n📊 Produits:      {diff_products:+,} ({diff_products/results_1['n_products']*100:+.1f}%)")
print(f"📈 Accuracy Test: {diff_acc:+.4f} ({diff_acc*100:+.2f}%)")
print(f"📈 F1-Score Test: {diff_f1:+.4f} ({diff_f1*100:+.2f}%)")
print(f"⏱️  Temps:         {diff_time:+.2f}s ({diff_time/results_1['best_metrics']['training_time']*100:+.1f}%)")

if diff_acc > 0:
    print(f"\n✅ Clean_2 est MEILLEUR en accuracy (+{diff_acc*100:.2f}%)")
elif diff_acc < 0:
    print(f"\n⚠️ Clean Original est MEILLEUR en accuracy ({diff_acc*100:.2f}%)")
else:
    print(f"\n➡️ Performance IDENTIQUE")

if diff_f1 > 0:
    print(f"✅ Clean_2 est MEILLEUR en F1-Score (+{diff_f1*100:.2f}%)")
elif diff_f1 < 0:
    print(f"⚠️ Clean Original est MEILLEUR en F1-Score ({diff_f1*100:.2f}%)")
else:
    print(f"➡️ F1-Score IDENTIQUE")

## 7. Visualisations Comparatives

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution Nutri-Score
grades = ['a', 'b', 'c', 'd', 'e']
dist1 = [results_1['distribution'].get(g, 0) for g in grades]
dist2 = [results_2['distribution'].get(g, 0) for g in grades]

x = np.arange(len(grades))
width = 0.35

axes[0, 0].bar(x - width/2, dist1, width, label='Clean Original', color='steelblue')
axes[0, 0].bar(x + width/2, dist2, width, label='Clean_2', color='forestgreen')
axes[0, 0].set_xlabel('Grade Nutri-Score')
axes[0, 0].set_ylabel('Nombre de produits')
axes[0, 0].set_title('Distribution du Nutri-Score', fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(grades)
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Performance Accuracy
metrics = ['Accuracy Test', 'F1-Score Test']
values1 = [results_1['best_metrics']['test_accuracy'], results_1['best_metrics']['test_f1']]
values2 = [results_2['best_metrics']['test_accuracy'], results_2['best_metrics']['test_f1']]

x = np.arange(len(metrics))
axes[0, 1].bar(x - width/2, values1, width, label='Clean Original', color='steelblue')
axes[0, 1].bar(x + width/2, values2, width, label='Clean_2', color='forestgreen')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_title('Performance des Meilleurs Modèles', fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(metrics)
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Comparaison XGBoost vs Random Forest
models = ['XGBoost', 'Random Forest']
acc1 = [results_1['xgb_metrics']['test_accuracy'], results_1['rf_metrics']['test_accuracy']]
acc2 = [results_2['xgb_metrics']['test_accuracy'], results_2['rf_metrics']['test_accuracy']]

x = np.arange(len(models))
axes[1, 0].bar(x - width/2, acc1, width, label='Clean Original', color='steelblue')
axes[1, 0].bar(x + width/2, acc2, width, label='Clean_2', color='forestgreen')
axes[1, 0].set_ylabel('Accuracy Test')
axes[1, 0].set_title('Comparaison par Modèle', fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(models)
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Temps d'entraînement
times1 = [results_1['xgb_metrics']['training_time'], results_1['rf_metrics']['training_time']]
times2 = [results_2['xgb_metrics']['training_time'], results_2['rf_metrics']['training_time']]

axes[1, 1].bar(x - width/2, times1, width, label='Clean Original', color='steelblue')
axes[1, 1].bar(x + width/2, times2, width, label='Clean_2', color='forestgreen')
axes[1, 1].set_ylabel('Temps (secondes)')
axes[1, 1].set_title('Temps d\'Entraînement', fontweight='bold')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(models)
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Tableau Récapitulatif Détaillé

In [ ]:
detailed_comparison = pd.DataFrame({
    'Modèle': ['XGBoost', 'XGBoost', 'Random Forest', 'Random Forest'],
    'Dataset': ['Clean Original', 'Clean_2', 'Clean Original', 'Clean_2'],
    'Accuracy Train': [
        f"{results_1['xgb_metrics']['train_accuracy']:.4f}",
        f"{results_2['xgb_metrics']['train_accuracy']:.4f}",
        f"{results_1['rf_metrics']['train_accuracy']:.4f}",
        f"{results_2['rf_metrics']['train_accuracy']:.4f}"
    ],
    'Accuracy Test': [
        f"{results_1['xgb_metrics']['test_accuracy']:.4f}",
        f"{results_2['xgb_metrics']['test_accuracy']:.4f}",
        f"{results_1['rf_metrics']['test_accuracy']:.4f}",
        f"{results_2['rf_metrics']['test_accuracy']:.4f}"
    ],
    'F1-Score Test': [
        f"{results_1['xgb_metrics']['test_f1']:.4f}",
        f"{results_2['xgb_metrics']['test_f1']:.4f}",
        f"{results_1['rf_metrics']['test_f1']:.4f}",
        f"{results_2['rf_metrics']['test_f1']:.4f}"
    ],
    'Temps (s)': [
        f"{results_1['xgb_metrics']['training_time']:.2f}",
        f"{results_2['xgb_metrics']['training_time']:.2f}",
        f"{results_1['rf_metrics']['training_time']:.2f}",
        f"{results_2['rf_metrics']['training_time']:.2f}"
    ]
})

print("\n" + "="*60)
print("TABLEAU DÉTAILLÉ - TOUS LES MODÈLES")
print("="*60)
print(f"\n{detailed_comparison.to_string(index=False)}")

## 9. Conclusion

In [ ]:
print("\n" + "="*60)
print("CONCLUSION")
print("="*60)

print(f"\n🏆 MEILLEUR MODÈLE GLOBAL:")

all_results = [
    ('Clean Original - XGBoost', results_1['xgb_metrics']['test_f1']),
    ('Clean Original - Random Forest', results_1['rf_metrics']['test_f1']),
    ('Clean_2 - XGBoost', results_2['xgb_metrics']['test_f1']),
    ('Clean_2 - Random Forest', results_2['rf_metrics']['test_f1'])
]

best_overall = max(all_results, key=lambda x: x[1])
print(f"   {best_overall[0]}")
print(f"   F1-Score: {best_overall[1]:.4f}")

print(f"\n📊 RECOMMANDATION:")
if results_2['best_metrics']['test_f1'] > results_1['best_metrics']['test_f1']:
    print(f"   ✅ Utiliser Clean_2 avec {results_2['best_name']}")
    print(f"   Gain: +{(results_2['best_metrics']['test_f1'] - results_1['best_metrics']['test_f1'])*100:.2f}% en F1-Score")
elif results_1['best_metrics']['test_f1'] > results_2['best_metrics']['test_f1']:
    print(f"   ✅ Utiliser Clean Original avec {results_1['best_name']}")
    print(f"   Gain: +{(results_1['best_metrics']['test_f1'] - results_2['best_metrics']['test_f1'])*100:.2f}% en F1-Score")
else:
    print(f"   ➡️ Performance identique, choisir selon la taille du dataset")

print("\n" + "="*60)